## WebScraping for balloting information

In [12]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
import numpy as np


YEARS = range(2009, 2026)
BASE = "https://sgschooling.com/year/{year}/all"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def clean_text(text: str) -> str:
    return " ".join(text.split())

def parse_regular_cell(text: str) -> int:
    """
    For Vacancy and Applied rows.
    Examples:
    '53' -> 53
    '-'  -> 0
    """
    s = clean_text(text)
    if not s or s == "-":
        return 0

    m = re.search(r"\b\d+\b", s)
    return int(m.group()) if m else 0

def parse_taken_cell(text: str) -> int:
    """
    For Taken rows.
    Examples:
    '65 SC1-2 17/12' -> 65
    '61 SC<1' -> 61
    '-' -> 0
    """
    s = clean_text(text)
    if not s or s == "-":
        return 0

    # first standalone integer is the phase-level taken count
    m = re.search(r"(?<!/)\b\d+\b(?!/)", s)
    if m:
        return int(m.group())

    # fallback if only ratio appears
    r = re.search(r"(\d+)\s*/\s*(\d+)", s)
    if r:
        return int(r.group(2))

    return 0

def detect_phase_layout(header_cells):
    """
    Two possible layouts:
    1. Phase 1, 2A, 2B, 2C, 2C(S), 3
    2. Phase 1, 2A(1), 2A(2), 2B, 2C, 2C(S), 3
    """
    cells = [clean_text(x) for x in header_cells]

    if "2A(1)" in cells and "2A(2)" in cells:
        return ["Phase 1", "2A(1)", "2A(2)", "2B", "2C", "2C(S)", "3"]
    return ["Phase 1", "2A", "2B", "2C", "2C(S)", "3"]

def extract_header_phases(soup):
    for tr in soup.select("tr"):
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
        if texts and texts[0] == "School":
            return detect_phase_layout(texts[1:])

    raise ValueError("Could not find header row.")

def is_school_row(first: str) -> bool:
    s = clean_text(first)

    if not s:
        return False
    if s.startswith("↳"):
        return False
    if s in {"School", "P1 Ballot History", "All Primary Schools"}:
        return False
    if "Vacancy" in s or "Applied" in s or "Taken" in s:
        return False

    return True

def extract_phase_values(texts, phases, row_type):
    """
    Map row values into a phase:value dictionary.
    """
    vals = {}
    raw_cells = texts[1:1 + len(phases)]

    for phase, cell in zip(phases, raw_cells):
        if row_type == "taken":
            vals[phase] = parse_taken_cell(cell)
        else:
            vals[phase] = parse_regular_cell(cell)

    for p in phases:
        vals.setdefault(p, 0)

    return vals

def parse_year(year: int):
    url = BASE.format(year=year)
    html = requests.get(url, headers=HEADERS, timeout=40).text
    soup = BeautifulSoup(html, "html.parser")

    phases = extract_header_phases(soup)

    out = []
    current_school = None
    vacancy_vals = None
    applied_vals = None
    taken_vals = None

    for tr in soup.select("tr"):
        cells = tr.find_all(["th", "td"])
        if not cells:
            continue

        texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]
        if not texts:
            continue

        first = texts[0]

        # school row
        if is_school_row(first):
            current_school = first
            vacancy_vals = None
            applied_vals = None
            taken_vals = None
            continue

        if not current_school:
            continue

        # vacancy row
        if "Vacancy" in first:
            vacancy_vals = extract_phase_values(texts, phases, row_type="vacancy")
            continue

        # applied row
        if "Applied" in first:
            applied_vals = extract_phase_values(texts, phases, row_type="applied")
            continue

        # taken row
        if "Taken" in first:
            taken_vals = extract_phase_values(texts, phases, row_type="taken")

            if vacancy_vals is None:
                vacancy_vals = {p: 0 for p in phases}
            if applied_vals is None:
                applied_vals = {p: 0 for p in phases}
            if taken_vals is None:
                taken_vals = {p: 0 for p in phases}

            # overall year ballot: 2B2C phase with applied > taken
            app_2B = applied_vals.get("2B", 0)
            app_2C = applied_vals.get("2C", 0)

            vac_2B = vacancy_vals.get("2B", 0)
            vac_2C = vacancy_vals.get("2C", 0)

            balloted_year = int(
                (app_2B > vac_2B) or (app_2C > vac_2C)
            )

            out.append({
                "school": current_school,
                "year": year,
                "applicants_2B": applied_vals.get("2B", 0),
                "applicants_2C": applied_vals.get("2C", 0),
                "vacancies_2B": vacancy_vals.get("2B", 0),
                "vacancies_2C": vacancy_vals.get("2C", 0),
                "taken_2B": taken_vals.get("2B", 0),
                "taken_2C": taken_vals.get("2C", 0),
                "balloted_year": balloted_year
            })

            vacancy_vals = None
            applied_vals = None
            taken_vals = None

    return out

# -------------------------
# Build full panel
# -------------------------
all_rows = []
for y in YEARS:
    rows = parse_year(y)
    print(f"{y}: {len(rows)} schools")
    all_rows.extend(rows)

df_2B2C_panel = pd.DataFrame(all_rows)

# remove duplicates if any
df_2B2C_panel = (
    df_2B2C_panel
    .groupby(["school", "year"], as_index=False)
    .first()
    .sort_values(["year", "school"])
    .reset_index(drop=True)
)

print(df_2B2C_panel.head(20))
print(df_2B2C_panel.shape)

# optional save
# df_2B2C_panel.to_csv("school_year_2B_2C_panel.csv", index=False)

2009: 168 schools
2010: 169 schools
2011: 170 schools
2012: 177 schools
2013: 180 schools
2014: 180 schools
2015: 183 schools
2016: 183 schools
2017: 184 schools
2018: 184 schools
2019: 185 schools
2020: 186 schools
2021: 181 schools
2022: 181 schools
2023: 181 schools
2024: 180 schools
2025: 179 schools
                     school  year  applicants_2B  applicants_2C  vacancies_2B  \
0                 Admiralty  2009             51            139            63   
1             Ahmad Ibrahim  2009              3             71            59   
2                   Ai Tong  2009             65             64            45   
3              Anchor Green  2009              0            110            95   
4                  Anderson  2009             25            116            57   
5                Ang Mo Kio  2009              2            107            65   
6    Anglo-Chinese (Junior)  2009             73             63            53   
7   Anglo-Chinese (Primary)  2009             

In [13]:
df_2B2C_panel

,school,year,applicants_2B,applicants_2C,vacancies_2B,vacancies_2C,taken_2B,taken_2C,balloted_year
0,Admiralty,2009,51,139,63,75,51,75,1
1,Ahmad Ibrahim,2009,3,71,59,116,3,70,0
2,Ai Tong,2009,65,64,45,48,45,48,1
3,Anchor Green,2009,0,110,95,189,0,110,0
4,Anderson,2009,25,116,57,88,25,88,1
...,...,...,...,...,...,...,...,...,...
3046,Yuhua,2025,1,26,58,174,1,26,0
3047,Yumin,2025,0,10,67,201,0,10,0
3048,Zhangde,2025,0,74,29,87,0,74,0
3049,Zhenghua,2025,0,50,20,62,0,50,0


In [14]:
df_2B2C_panel["RBF_st"] = (
    df_2B2C_panel.groupby("school")["balloted_year"]
      .transform(lambda s: s.rolling(window=5, min_periods=1).mean())
)

In [15]:
df_2B2C_panel['BI'] = np.log((df_2B2C_panel['applicants_2B']+df_2B2C_panel['applicants_2C'])/(df_2B2C_panel['vacancies_2B']+df_2B2C_panel['vacancies_2C']))

In [16]:
df_2B2C_panel["across_phase_ballot"] = (
    (df_2B2C_panel["applicants_2B"] > df_2B2C_panel["vacancies_2B"]) &
    (df_2B2C_panel["applicants_2C"] > df_2B2C_panel["vacancies_2C"])
).astype(int)

In [17]:
#We should start from 2013 omwards so that there is the full 5 years for each school year
df_2B2C_panel = df_2B2C_panel[df_2B2C_panel['year']>=2013]


In [18]:
#Normalise 
from scipy.stats import zscore
df_2B2C_panel["z_BI"] = 0.7 * zscore(
    df_2B2C_panel["BI"].astype(float),
    nan_policy="omit"
) + 0.3 * df_2B2C_panel['across_phase_ballot']

df_2B2C_panel['z_RBF'] = zscore(
    df_2B2C_panel["RBF_st"].astype(float),
    nan_policy="omit"
)

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1155/4206131706.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel["z_BI"] = 0.7 * zscore(
/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1155/4206131706.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel['z_RBF'] = zscore(


In [19]:
df_2B2C_panel['SDI']= (df_2B2C_panel['z_RBF']+df_2B2C_panel['z_BI'])/2

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1155/3049317479.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_2B2C_panel['SDI']= (df_2B2C_panel['z_RBF']+df_2B2C_panel['z_BI'])/2


In [20]:
df_2B2C_panel

,school,year,applicants_2B,applicants_2C,vacancies_2B,vacancies_2C,taken_2B,taken_2C,balloted_year,RBF_st,BI,across_phase_ballot,z_BI,z_RBF,SDI
684,Admiralty,2013,40,96,63,86,40,86,1,1.0,-0.091291,0,0.337804,1.110344,0.724074
685,Ahmad Ibrahim,2013,2,73,60,118,2,73,0,0.0,-0.864295,0,-0.275518,-1.096885,-0.686201
686,Ai Tong,2013,25,38,8,7,8,7,1,1.0,1.435085,1,1.848872,1.110344,1.479608
687,Alexandra,2013,0,232,105,210,0,210,1,1.0,-0.305835,0,0.167580,1.110344,0.638962
688,Anchor Green,2013,3,90,41,78,3,78,1,0.4,-0.246524,0,0.214639,-0.213993,0.000323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3046,Yuhua,2025,1,26,58,174,1,26,0,0.0,-2.150901,0,-1.296345,-1.096885,-1.196615
3047,Yumin,2025,0,10,67,201,0,10,0,0.0,-3.288402,0,-2.198870,-1.096885,-1.647877
3048,Zhangde,2025,0,74,29,87,0,74,0,0.2,-0.449525,0,0.053572,-0.655439,-0.300933
3049,Zhenghua,2025,0,50,20,62,0,50,0,0.0,-0.494696,0,0.017732,-1.096885,-0.539576


In [21]:
SDI_df = df_2B2C_panel[['school','year','SDI']]

## Web Scraping for GEP/SAP information

In [43]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
)

# =========================
# CONFIG
# =========================

URL = "https://www.property2b2c.com/school-ranking/bespoke"
YEARS = list(range(2013, 2026))

# =========================
# BROWSER SETUP
# =========================

options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless=new")  # uncomment if needed

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# =========================
# HELPERS
# =========================

def safe_click(el):
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    time.sleep(0.4)
    try:
        el.click()
    except Exception:
        driver.execute_script("arguments[0].click();", el)

def open_year_modal():
    """
    Click the top-right chip like 'SC | 2019'
    """
    candidate_xpaths = [
        "//div[contains(@class,'MuiChip-root')]//button",
        "//span[contains(., '2019')]/ancestor::div[contains(@class,'MuiChip-root')]//button",
        "//*[contains(text(),'2019')]/ancestor::div[contains(@class,'MuiChip-root')]//button",
    ]

    for xp in candidate_xpaths:
        elems = driver.find_elements(By.XPATH, xp)
        for el in elems:
            try:
                if el.is_displayed() and el.is_enabled():
                    safe_click(el)
                    # wait for modal
                    wait.until(
                        EC.presence_of_element_located(
                            (By.XPATH, "//button[@role='combobox' and @name='from']")
                        )
                    )
                    wait.until(
                        EC.presence_of_element_located(
                            (By.XPATH, "//button[@role='combobox' and @name='to']")
                        )
                    )
                    time.sleep(0.8)
                    return
            except StaleElementReferenceException:
                continue

    raise Exception("Could not open year modal.")

def get_combobox(name):
    """
    name = 'from' or 'to'
    """
    return wait.until(
        EC.presence_of_element_located(
            (By.XPATH, f"//button[@role='combobox' and @name='{name}']")
        )
    )

def open_combobox(name):
    """
    Open the combobox and wait until aria-expanded='true'
    """
    btn = get_combobox(name)
    safe_click(btn)

    wait.until(
        lambda d: d.find_element(
            By.XPATH,
            f"//button[@role='combobox' and @name='{name}']"
        ).get_attribute("aria-expanded") == "true"
    )

    time.sleep(0.5)
    return get_combobox(name)

def select_option_for_combobox(name, year):
    """
    Uses aria-controls from the combobox to target the correct dropdown list.
    This is the key fix for the 'to' year not being selected.
    """
    year = str(year)

    # open the combobox first
    btn = open_combobox(name)

    controls_id = btn.get_attribute("aria-controls")
    if not controls_id:
        raise Exception(f"Combobox '{name}' has no aria-controls attribute.")

    # Try several possible structures for the controlled popup/listbox
    option_xpaths = [
        f"//*[@id='{controls_id}']//li[@role='option' and normalize-space(.)='{year}']",
        f"//*[@id='{controls_id}']//*[@role='option' and normalize-space(.)='{year}']",
        f"//li[@role='option' and normalize-space(.)='{year}' and ancestor::*[@id='{controls_id}']]",
    ]

    last_error = None

    for xp in option_xpaths:
        try:
            wait.until(EC.presence_of_element_located((By.XPATH, xp)))
            options = driver.find_elements(By.XPATH, xp)

            for opt in options:
                try:
                    if opt.is_displayed():
                        safe_click(opt)
                        time.sleep(0.8)

                        # wait until combobox closes
                        wait.until(
                            lambda d: d.find_element(
                                By.XPATH,
                                f"//button[@role='combobox' and @name='{name}']"
                            ).get_attribute("aria-expanded") == "false"
                        )

                        # verify selected value changed
                        value_input = driver.find_element(
                            By.XPATH,
                            f"//input[@name='{name}']"
                        )
                        selected_value = value_input.get_attribute("value")
                        if selected_value != year:
                            raise Exception(
                                f"Selected option clicked but {name} value is still '{selected_value}', expected '{year}'."
                            )

                        return

                except StaleElementReferenceException as e:
                    last_error = e
                    continue
                except Exception as e:
                    last_error = e
                    continue

        except Exception as e:
            last_error = e
            continue

    raise Exception(f"Could not select year {year} for '{name}'. Last error: {last_error}")

def click_apply():
    apply_btn = wait.until(
        EC.element_to_be_clickable(
            (By.XPATH, "//button[@type='submit' and normalize-space(.)='Apply']")
        )
    )
    safe_click(apply_btn)
    time.sleep(1.5)

    # wait until modal disappears
    end_time = time.time() + 10
    while time.time() < end_time:
        apply_buttons = driver.find_elements(
            By.XPATH, "//button[@type='submit' and normalize-space(.)='Apply']"
        )
        visible = False
        for b in apply_buttons:
            try:
                if b.is_displayed():
                    visible = True
                    break
            except Exception:
                pass
        if not visible:
            return
        time.sleep(0.3)

def get_table_rows():
    candidate_xpaths = [
        "//table//tbody//tr",
        "//table//tr[td]",
        "//div[contains(@class,'MuiTableBody-root')]//div[contains(@class,'MuiTableRow-root')]",
        "//div[contains(@class,'MuiTable-root')]//div[contains(@class,'MuiTableRow-root')]",
    ]

    for xp in candidate_xpaths:
        rows = driver.find_elements(By.XPATH, xp)
        visible_rows = []
        for r in rows:
            try:
                if r.is_displayed() and r.text.strip():
                    visible_rows.append(r)
            except StaleElementReferenceException:
                continue

        if visible_rows:
            return visible_rows

    raise Exception("Could not find visible table rows.")

def extract_school_from_row(row):
    candidate_xpaths = [
        ".//a",
        ".//*[contains(@class,'MuiLink-root')]",
        ".//td[2]",
    ]

    for xp in candidate_xpaths:
        try:
            elems = row.find_elements(By.XPATH, xp)
            for el in elems:
                txt = el.text.strip()
                if txt and txt.lower() != "school":
                    return txt
        except Exception:
            continue

    return None

def parse_current_table(year):
    rows = get_table_rows()
    data = []

    for r in rows:
        try:
            school = extract_school_from_row(r)
            if not school:
                continue

            row_text = r.text.upper()
            gep = 1 if ("GEP" in row_text and "GEP2" not in row_text) else 0
            sap = 1 if ("SAP" in row_text) else 0

            data.append({
                "school": school,
                "year": year,
                "GEP": gep,
                "SAP": sap
            })

        except Exception:
            continue

    return pd.DataFrame(data).drop_duplicates(subset=["school", "year"]).reset_index(drop=True)

def wait_for_table_refresh(previous_first_school=None, timeout=20):
    end_time = time.time() + timeout

    while time.time() < end_time:
        try:
            rows = get_table_rows()
            if rows:
                first_school = extract_school_from_row(rows[0])
                if previous_first_school is None:
                    return
                if first_school and first_school != previous_first_school:
                    return
        except Exception:
            pass

        time.sleep(0.6)

    return

def set_single_year(year):
    """
    Sets from=year and to=year, then clicks Apply.
    """
    open_year_modal()

    select_option_for_combobox("from", year)
    select_option_for_combobox("to", year)

    # verify before applying
    from_value = driver.find_element(By.XPATH, "//input[@name='from']").get_attribute("value")
    to_value = driver.find_element(By.XPATH, "//input[@name='to']").get_attribute("value")

    if from_value != str(year):
        raise Exception(f"'from' did not update correctly. Expected {year}, got {from_value}")

    if to_value != str(year):
        raise Exception(f"'to' did not update correctly. Expected {year}, got {to_value}")

    click_apply()

# =========================
# RUN
# =========================

all_dfs = []

try:
    driver.get(URL)
    time.sleep(5)

    previous_first_school = None

    for year in YEARS:
        print(f"Scraping {year}...")

        set_single_year(year)
        wait_for_table_refresh(previous_first_school=previous_first_school, timeout=20)

        df_year = parse_current_table(year)
        print(f"  rows: {len(df_year)}")

        if not df_year.empty:
            previous_first_school = df_year.iloc[0]["school"]
            all_dfs.append(df_year)

finally:
    driver.quit()

# =========================
# FINAL OUTPUT
# =========================

if all_dfs:
    final = pd.concat(all_dfs, ignore_index=True)
    final = final.drop_duplicates(subset=["school", "year"]).sort_values(["year", "school"]).reset_index(drop=True)
    final.to_csv("property2b2c_school_year_gep_sap.csv",index=False)
else:
    print("No data scraped.")

Scraping 2013...
  rows: 176
Scraping 2014...
  rows: 176
Scraping 2015...
  rows: 179
Scraping 2016...
  rows: 179
Scraping 2017...
  rows: 180
Scraping 2018...
  rows: 180
Scraping 2019...
  rows: 181
Scraping 2020...
  rows: 182
Scraping 2021...
  rows: 181
Scraping 2022...
  rows: 181
Scraping 2023...
  rows: 181
Scraping 2024...
  rows: 180
Scraping 2025...
  rows: 179


## Dataframe Manipulation

In [45]:
# Some changes to the names to merge correctly
SDI_df['school'] = SDI_df['school'].str.replace("’","'", regex=False)
final['school'] = final['school'].str.replace("’","'", regex=False)
SDI_df['school'] = SDI_df['school'].str.replace("Anglo-Chinese (Primary)","Anglo-Chinese")

/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1155/1093316296.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  SDI_df['school'] = SDI_df['school'].str.replace("’","'", regex=False)
/var/folders/2m/8llkplds47v3b1fv6f6vqv0m0000gn/T/ipykernel_1155/1093316296.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  SDI_df['school'] = SDI_df['school'].str.replace("Anglo-Chinese (Primary)","Anglo-Chinese")


In [46]:
full_df = SDI_df.merge(final, how="left", left_on= ['school','year'], right_on=['school','year'])

In [47]:
full_df

,school,year,SDI,GEP,SAP
0,Admiralty,2013,0.724074,0.0,0.0
1,Ahmad Ibrahim,2013,-0.686201,0.0,0.0
2,Ai Tong,2013,1.479608,0.0,1.0
3,Alexandra,2013,0.638962,0.0,0.0
4,Anchor Green,2013,0.000323,0.0,0.0
...,...,...,...,...,...
2362,Yuhua,2025,-1.196615,0.0,0.0
2363,Yumin,2025,-1.647877,0.0,0.0
2364,Zhangde,2025,-0.300933,0.0,0.0
2365,Zhenghua,2025,-0.539576,0.0,0.0


In [48]:
full_df[full_df['school'].isnull()]

,school,year,SDI,GEP,SAP


In [49]:
full_df['GSI']= 0.6 * zscore(
    full_df["SDI"].astype(float),
    nan_policy="omit"
) + 0.2 * full_df['GEP'] + 0.2 * full_df['SAP']

In [50]:
top20 = (
    full_df.sort_values(["year", "GSI"], ascending=[True, False])
      .groupby("year")
      .head(20)
      .reset_index(drop=True)
)

In [51]:
top20.to_csv("top20_GSI.csv", index=False)

In [52]:
bottom_20  = (
    df_2B2C_panel.sort_values(["year", "SDI"], ascending=[True, True])
      .groupby("year")
      .head(20)
      .reset_index(drop=True)
)

In [53]:
bottom_20.to_csv("bottom_20_GSI.csv", index=False)

In [54]:
full_df

,school,year,SDI,GEP,SAP,GSI
0,Admiralty,2013,0.724074,0.0,0.0,0.499763
1,Ahmad Ibrahim,2013,-0.686201,0.0,0.0,-0.503929
2,Ai Tong,2013,1.479608,0.0,1.0,1.237476
3,Alexandra,2013,0.638962,0.0,0.0,0.439189
4,Anchor Green,2013,0.000323,0.0,0.0,-0.015330
...,...,...,...,...,...,...
2362,Yuhua,2025,-1.196615,0.0,0.0,-0.867190
2363,Yumin,2025,-1.647877,0.0,0.0,-1.188353
2364,Zhangde,2025,-0.300933,0.0,0.0,-0.229734
2365,Zhenghua,2025,-0.539576,0.0,0.0,-0.399576


In [55]:
# thresholds to test
thresholds = [0.75, 0.80, 0.85]

for q in thresholds:
    pct = int(q * 100)
    
    # year-specific cutoff
    full_df[f"GSI_p{pct}_year"] = (
        full_df.groupby("year")["GSI"]
        .transform(lambda x: x.quantile(q))
    )
    
    # indicator for whether school is at or above that cutoff
    full_df[f"good_school_{pct}"] = (
        full_df["GSI"] >= full_df[f"GSI_p{pct}_year"]
    ).astype(int)

In [56]:
full_df

,school,year,SDI,GEP,SAP,GSI,GSI_p75_year,good_school_75,GSI_p80_year,good_school_80,GSI_p85_year,good_school_85
0,Admiralty,2013,0.724074,0.0,0.0,0.499763,0.546139,0,0.613108,0,0.713556,0
1,Ahmad Ibrahim,2013,-0.686201,0.0,0.0,-0.503929,0.546139,0,0.613108,0,0.713556,0
2,Ai Tong,2013,1.479608,0.0,1.0,1.237476,0.546139,1,0.613108,1,0.713556,1
3,Alexandra,2013,0.638962,0.0,0.0,0.439189,0.546139,0,0.613108,0,0.713556,0
4,Anchor Green,2013,0.000323,0.0,0.0,-0.015330,0.546139,0,0.613108,0,0.713556,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2362,Yuhua,2025,-1.196615,0.0,0.0,-0.867190,0.623686,0,0.729251,0,0.799601,0
2363,Yumin,2025,-1.647877,0.0,0.0,-1.188353,0.623686,0,0.729251,0,0.799601,0
2364,Zhangde,2025,-0.300933,0.0,0.0,-0.229734,0.623686,0,0.729251,0,0.799601,0
2365,Zhenghua,2025,-0.539576,0.0,0.0,-0.399576,0.623686,0,0.729251,0,0.799601,0


In [57]:
last_df = full_df[['school', 'year', 'GSI','good_school_75','good_school_80','good_school_85']]

In [58]:
last_df

,school,year,GSI,good_school_75,good_school_80,good_school_85
0,Admiralty,2013,0.499763,0,0,0
1,Ahmad Ibrahim,2013,-0.503929,0,0,0
2,Ai Tong,2013,1.237476,1,1,1
3,Alexandra,2013,0.439189,0,0,0
4,Anchor Green,2013,-0.015330,0,0,0
...,...,...,...,...,...,...
2362,Yuhua,2025,-0.867190,0,0,0
2363,Yumin,2025,-1.188353,0,0,0
2364,Zhangde,2025,-0.229734,0,0,0
2365,Zhenghua,2025,-0.399576,0,0,0
